#  Dataprocessing of datasets - counts
This notebook filters GTEx and TCGA expression data to include only **pancreas samples**, keeping gene identifiers and harmonizing formats for downstream CSD-analyzing. The functions used in this pipeline is found in ../../utils.

**Goal:** Load, clean, normalize, transform and prepare RNA-seq data for multi-omics integration (CSD network analysis)

## Imports and files
Imports needed for all data processing:

In [1]:
import sys
from pathlib import Path

# Add project scripts to path
sys.path.insert(0, str(Path("../../..").resolve() / "scripts"))

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from utils.rna_processing import (
    load_gtex,
    load_tcga_tpm,
    process_tcga_data,
    clean_df,
    load_protein_coding_genes,
    filter_protein_coding,
    filter_low_expression_counts,
    harmonize_datasets,
    plot_pca,
    prepare_findcorrvar_input,
)


In [13]:
# Directory paths
DATA_DIR = Path("../../../data/raw/")
OUT_DIR = Path("../../../data/processed/")
RESULT_DIR = Path("../../../results/")

# Raw data
gtex_gct = DATA_DIR / "gtex/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct"
gtex_meta = DATA_DIR / "gtex/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
tcga_samples = DATA_DIR / "gdc_sample_sheet.tsv"
tcga_counts_dir = DATA_DIR / "tcga/"


# Processed data output paths
gtex_output_path = OUT_DIR / "gtex/gtex_pancreas_processed.csv"
tcga_output_path = OUT_DIR / "tcga/tcga_pancreas_processed.csv"

# Paths for data integration
gtf_path = DATA_DIR / "gencode/gencode.v47.genes.gtf"
gtex_out_int = OUT_DIR / "gtex/gtex_int.csv"
tcga_out_int = OUT_DIR / "tcga/tcga_int.csv"
mapping_file = OUT_DIR / "genelist/common_protein_coding_genes.txt"
proteome_common_genes = OUT_DIR / "genelist/proteome_common_genes.txt"


# Paths for final file
out_gtex_final = OUT_DIR / "rna_healthy.txt"
out_tcga_final = OUT_DIR / "rna_cancer.txt"

## Main pipeline

### Load and clean data



From raw data to clean .csv datasets:

GTEX raw tpm -> GTEX only pancreas genes -> 

In [14]:
# Process GTEx data
gtex_raw = load_gtex(gtex_meta, gtex_gct)


C:\Users\tiril\Master-Tiril\scripts\utils\rna_processing.py:25: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(gtex_meta, sep="\t", low_memory=False)


Found 961 pancreas samples in GTEx metadata.
Loading only 384 pancreas columns
Loaded GTEx subset with shape: (74628, 385)


In [15]:
# Process TCGA data
tcga_raw = process_tcga_data(tcga_counts_dir, tcga_samples, value_col="unstranded")


Loaded 183 sample mappings from gdc_sample_sheet.tsv
Found 183 STAR counts files in ..\..\..\data\raw\tcga.
Merged 183 samples into a single DataFrame with shape: (60660, 184)


In [16]:
# Clean datasets
gtex_clean = clean_df(gtex_raw)
tcga_clean = clean_df(tcga_raw)

Original data shape: (74628, 385)
Removed 0 duplicate genes.
Data shape after removing duplicates: (74628, 385)
Original data shape: (60660, 184)
Removed 44 duplicate genes.
Data shape after removing duplicates: (60616, 184)


### Data integration

In [17]:
# Filter out PC genes
pc_genes = load_protein_coding_genes(gtf_path, out_file=mapping_file)[0]
gtex_pc_filt= filter_protein_coding(gtex_clean, pc_genes)
tcga_pc_filt = filter_protein_coding(tcga_clean, pc_genes)



Extracted 19355 protein-coding genes from ..\..\..\data\raw\gencode\gencode.v47.genes.gtf
Saved gene ID to name mapping to ..\..\..\data\processed\genelist\common_protein_coding_genes.txt
Filtered to protein-coding genes: (19355, 385)
Filtered to protein-coding genes: (19299, 184)


In [18]:
gtex_filt = filter_low_expression_counts(gtex_pc_filt, min_cpm=1, min_fraction=0.2)
tcga_filt = filter_low_expression_counts(tcga_pc_filt, min_cpm=1, min_fraction=0.2)


Original genes: 19355
Genes retained: (13833, 385)
Minimum samples required: 77
Original genes: 19299
Genes retained: (14993, 184)
Minimum samples required: 37


In [19]:
# Harmonize genelist
gtex_harm, tcga_harm = harmonize_datasets(gtex_filt, tcga_filt, proteome_common_genes, out_dir=OUT_DIR)

gtex_harm.to_csv("../../../data/processed/gtex/gtex_counts_harm.csv", index=False)
tcga_harm.to_csv("../../../data/processed/tcga/tcga_counts_harm.csv", index=False)


Number of common protein-coding genes: 7964
Shapes after filtering to common genes — GTEx: (7964, 385), TCGA: (7964, 184)


### Data Transformation: 

#### DeSeq2 transformasjon

In [20]:
gtex_deseq = pd.read_csv("../../../data/processed/gtex/gtex_vst.csv", index_col=0)
tcga_deseq = pd.read_csv("../../../data/processed/tcga/tcga_vst.csv", index_col=0)

gtex_deseq = gtex_deseq.reset_index().rename(columns={"index": "gene_id"})

tcga_deseq = tcga_deseq.reset_index().rename(columns={"index": "gene_id"})

print(gtex_deseq.shape)
print(tcga_deseq.shape)



(7964, 385)
(7964, 184)


## Save section

In [12]:
# Save loaded and processed data
gtex_raw.to_csv(gtex_output_path, index=False)
print(f"Processed GTEx pancreas data saved to {gtex_output_path}")
tcga_raw.to_csv(tcga_output_path, index=False)
print(f"Processed TCGA data saved to {tcga_output_path}")

# Save final dataset with gene_id and original sample IDs as header
gtex_deseq.to_csv(out_gtex_final, header=True, index=False)
print(f"Final dataset for GTEX is saved to {out_gtex_final}")
tcga_deseq.to_csv(out_tcga_final, header=True, index=False)
print(f"Final dataset for TCGA is saved to {out_tcga_final}")

Processed GTEx pancreas data saved to ..\..\..\data\processed\gtex\gtex_pancreas_processed.csv
Processed TCGA data saved to ..\..\..\data\processed\tcga\tcga_pancreas_processed.csv
Final dataset for GTEX is saved to ..\..\..\data\processed\rna_healthy.txt
Final dataset for TCGA is saved to ..\..\..\data\processed\rna_cancer.txt


In [ ]:
# Converts gene expression files to findcorrvar-compatible format
prepare_findcorrvar_input(out_gtex_final, out=OUT_DIR / "ExpData_rna_Healthy.txt", float_format="{:.6f}")
prepare_findcorrvar_input(out_tcga_final, out=OUT_DIR / "ExpData_rna_Cancer.txt", float_format="{:.6f}")
